In [ ]:
import pandas as pd
import snowflake.connector
import getpass

ACCOUNT  = "PMGPITV-ZI03016"
USER     = "KATMLACKA"
PASSWORD = getpass.getpass("Hasło Snowflake: ")

conn = snowflake.connector.connect(
    account   = ACCOUNT,
    user      = USER,
    password  = PASSWORD,
    warehouse = "COMPUTE_WH",
    database  = "TRIAL_DB",
    schema    = "PUBLIC",
    autocommit=True
)
cursor = conn.cursor()
print("Połączono ze Snowflake.")

df_orders    = pd.read_excel("data.xlsx", sheet_name="orders")
df_statuses  = pd.read_excel("data.xlsx", sheet_name="order_statuses")

cursor.execute("SELECT ORDER_ID FROM ORDERS")
istniejace = set(row[0] for row in cursor.fetchall())

df_orders_nowe = df_orders[~df_orders["order_id"].isin(istniejace)]
df_statuses_nowe = df_statuses[~df_statuses["order_id"].isin(istniejace)]

for _, row in df_orders_nowe.iterrows():
    cursor.execute("""
        INSERT INTO ORDERS (
            ORDER_ID, CLIENT_ID, CLIENT_NAME, CARRIER_ID, CARRIER_NAME, VEHICLE_ID,
            LOADING_DATE, ACTUAL_LOADING_DATE, UNLOADING_DATE, ACTUAL_UNLOADING_DATE,
            LOADING_LOCATION_ID, LOADING_PLACE_NAME, LOADING_PLACE_STREET,
            LOADING_PLACE_POSTALCODE, LOADING_PLACE_CITY, LOADING_PLACE_COUNTRY,
            UNLOADING_LOCATION_ID, UNLOADING_PLACE_NAME, UNLOADING_PLACE_STREET,
            UNLOADING_PLACE_POSTALCODE, UNLOADING_PLACE_CITY, UNLOADING_PLACE_COUNTRY,
            WEIGHT, VOLUME, PALLET, TRUCK_TYPE, TRUCK_CAPACITY_WEIGHT, TRUCK_CAPACITY_VOLUME
        ) VALUES (
            %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s
        )
    """, (
        str(row["order_id"]), str(row["client_id"]), str(row["client_name"]),
        str(row["carrier_id"]), str(row["carrier_name"]), str(row["vehicle_id"]),
        str(row["loading_date"]), str(row["actual_loading_date"]),
        str(row["unloading_date"]), str(row["actual_unloading_date"]),
        str(row["loading_location_id"]), str(row["loading_place_name"]),
        str(row["loading_place_street"]), str(row["loading_place_postalcode"]),
        str(row["loading_place_city"]), str(row["loading_place_country"]),
        str(row["unloading_location_id"]), str(row["unloading_place_name"]),
        str(row["unloading_place_street"]), str(row["unloading_place_postalcode"]),
        str(row["unloading_place_city"]), str(row["unloading_place_country"]),
        float(row["weight"]), float(row["volume"]), int(row["pallet"]),
        str(row["truck_type"]), float(row["truck_capacity_weight"]),
        float(row["truck_capacity_volume"])
    ))

for _, row in df_statuses_nowe.iterrows():
    cursor.execute("""
        INSERT INTO ORDER_STATUSES (ORDER_ID, STATUS, STATUS_TIMESTAMP, REASON_CODE, REASON_DESC)
        VALUES (%s, %s, %s, %s, %s)
    """, (
        str(row["order_id"]),
        str(row["status"]),
        str(row["status_timestamp"]),
        None if pd.isna(row["reason_code"]) else str(row["reason_code"]),
        None if pd.isna(row["reason_desc"]) else str(row["reason_desc"])
    ))


cursor.execute("SELECT COUNT(*) FROM ORDERS")
conn.close()


In [ ]:
import random
import time
import pandas as pd
import snowflake.connector
import getpass
from datetime import datetime, timedelta

ACCOUNT  = "PMGPITV-ZI03016"
USER     = "KATMLACKA"
PASSWORD = getpass.getpass("Hasło Snowflake: ")

conn = snowflake.connector.connect(
    account   = ACCOUNT, user = USER, password = PASSWORD,
    warehouse = "COMPUTE_WH", database = "TRIAL_DB", schema = "PUBLIC",
    autocommit=True
)
cursor = conn.cursor()


df_clients   = pd.read_excel("data.xlsx", sheet_name="clients")
df_carriers  = pd.read_excel("data.xlsx", sheet_name="carriers")
df_vehicles  = pd.read_excel("data.xlsx", sheet_name="vehicles")
df_locations = pd.read_excel("data.xlsx", sheet_name="locations")

ZASIEG_KLIENTA = {"CL001": ["WH-001", "WH-002"], "CL002": ["WH-003"], "CL003": ["WH-004"]}
DESTYNACJE_KLIENTOW = {
    "CL001": ["LOC-001","LOC-002","LOC-003","LOC-004","LOC-005","LOC-006","LOC-011","LOC-012"],
    "CL002": ["LOC-014","LOC-015","LOC-016","LOC-007","LOC-008","LOC-013"],
    "CL003": ["LOC-017","LOC-018","LOC-019","LOC-020","LOC-009","LOC-010"]
}
REASON_CODES_MIEDZYNARODOWE = [("T01", "Opóźnienie na granicy"), ("T02", "Korek / warunki drogowe"), ("T03", "Awaria pojazdu"), ("T05", "Warunki pogodowe")]
REASON_CODES_ZAŁADUNEK = [("T04", "Opóźnienie załadunku przez klienta"), ("T08", "Brak towaru w magazynie")]

SEKWENCJA = {"NOWE": "POTWIERDZONE", "POTWIERDZONE": "ZAŁADUNEK", "ZAŁADUNEK": "W_TRANSPORCIE", "W_TRANSPORCIE": "ROZŁADUNEK", "ROZŁADUNEK": "DOSTARCZONE", "OPOZNIENIE_ZAŁADUNEK": "ZAŁADUNEK", "OPOZNIENIE_ROZŁADUNEK": "ROZŁADUNEK", "DOSTARCZONE": None}

cursor.execute("SELECT DISTINCT ORDER_ID FROM ORDERS")
wygenerowane_id = [row[0] for row in cursor.fetchall()]
numer_porzadkowy = 3001 if len(wygenerowane_id) == 0 else int(max(wygenerowane_id).split('-')[1]) + 1

licznik_petli = 0

while True:
    licznik_petli += 1
    currentTime = datetime.now()
    
    if licznik_petli % 3 == 0:
        nowy_id = f"ORD-{str(numer_porzadkowy).zfill(5)}"
        wygenerowane_id.append(nowy_id)
        
        klient = df_clients.sample(1).iloc[0]
        przewoznik = df_carriers[df_carriers["client_id"] == klient["client_id"]].sample(1).iloc[0]
        pojazd = df_vehicles[df_vehicles["carrier_id"] == przewoznik["carrier_id"]].sample(1).iloc[0]
        miejsce_zal = df_locations[(df_locations["location_type"] == "WAREHOUSE") & (df_locations["location_id"].isin(ZASIEG_KLIENTA[klient["client_id"]]))].sample(1).iloc[0]
        miejsce_rozl = df_locations[df_locations["location_id"].isin(DESTYNACJE_KLIENTOW[klient["client_id"]])].sample(1).iloc[0]

        loading_date   = currentTime + timedelta(minutes=random.randint(60, 480))
        unloading_date = loading_date + timedelta(minutes=random.randint(120, 360))
        weight = round(pojazd["capacity_weight"] * random.uniform(0.5, 0.98), 0)
        volume = round(pojazd["capacity_volume"] * random.uniform(0.5, 0.98), 1)
        pallet = random.randint(1, int(pojazd["capacity_pallet"]))

        cursor.execute("""
            INSERT INTO ORDERS (ORDER_ID, CLIENT_ID, CLIENT_NAME, CARRIER_ID, CARRIER_NAME, VEHICLE_ID, LOADING_DATE, ACTUAL_LOADING_DATE, UNLOADING_DATE, ACTUAL_UNLOADING_DATE, LOADING_LOCATION_ID, LOADING_PLACE_NAME, LOADING_PLACE_STREET, LOADING_PLACE_POSTALCODE, LOADING_PLACE_CITY, LOADING_PLACE_COUNTRY, UNLOADING_LOCATION_ID, UNLOADING_PLACE_NAME, UNLOADING_PLACE_STREET, UNLOADING_PLACE_POSTALCODE, UNLOADING_PLACE_CITY, UNLOADING_PLACE_COUNTRY, WEIGHT, VOLUME, PALLET, TRUCK_TYPE, TRUCK_CAPACITY_WEIGHT, TRUCK_CAPACITY_VOLUME)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """, (nowy_id, klient["client_id"], klient["client_name"], przewoznik["carrier_id"], przewoznik["carrier_name"], pojazd["vehicle_id"], loading_date.strftime('%Y-%m-%d %H:%M:%S'), loading_date.strftime('%Y-%m-%d %H:%M:%S'), unloading_date.strftime('%Y-%m-%d %H:%M:%S'), unloading_date.strftime('%Y-%m-%d %H:%M:%S'), miejsce_zal["location_id"], miejsce_zal["location_name"], miejsce_zal["street"], miejsce_zal["postal_code"], miejsce_zal["city"], miejsce_zal["country"], miejsce_rozl["location_id"], miejsce_rozl["location_name"], miejsce_rozl["street"], miejsce_rozl["postal_code"], miejsce_rozl["city"], miejsce_rozl["country"], weight, volume, pallet, pojazd["truck_type"], pojazd["capacity_weight"], pojazd["capacity_volume"]))

        cursor.execute("INSERT INTO ORDER_STATUSES (ORDER_ID, STATUS, STATUS_TIMESTAMP) VALUES (%s, 'NOWE', %s)", (nowy_id, currentTime.strftime('%Y-%m-%d %H:%M:%S')))
        numer_porzadkowy += 1

    if len(wygenerowane_id) > 0:
        order_id = random.choice(wygenerowane_id)
        cursor.execute("SELECT STATUS FROM ORDER_STATUSES WHERE ORDER_ID = %s ORDER BY STATUS_TIMESTAMP DESC LIMIT 1", (order_id,))
        row = cursor.fetchone()
        aktualny = row[0] if row else "NOWE"
        nastepny = SEKWENCJA.get(aktualny)
        
        if nastepny is not None:
           
            if aktualny == "W_TRANSPORCIE" and random.random() < 0.12: nastepny = "OPOZNIENIE_ROZŁADUNEK"
            if aktualny == "ZAŁADUNEK" and random.random() < 0.12: nastepny = "OPOZNIENIE_ZAŁADUNEK"
            reason = random.choice(REASON_CODES_ZAŁADUNEK if nastepny == "OPOZNIENIE_ZAŁADUNEK" else REASON_CODES_MIEDZYNARODOWE) if nastepny in ["OPOZNIENIE_ZAŁADUNEK", "OPOZNIENIE_ROZŁADUNEK"] else (None, None)
            
            cursor.execute("INSERT INTO ORDER_STATUSES (ORDER_ID, STATUS, STATUS_TIMESTAMP, REASON_CODE, REASON_DESC) VALUES (%s, %s, %s, %s, %s)", (order_id, nastepny, currentTime.strftime('%Y-%m-%d %H:%M:%S'), reason[0], reason[1]))
            
    time.sleep(5)